# Milestone 2 — Resume Parsing: Dataset Pipeline

Single reproducible notebook for the whole processing. Run top-to-bottom from the
`Milestone 2/` directory (needs the raw corpus in `./extracted/Resumes PDF`).

**Part 1 — EDA:** structural scan, OCR text-length sample, summary numbers, figures.
**Part 2 — Preparation:** near-duplicate detection (perceptual hash), stratified
gold-set sampling, dev/test split, and building `final_dataset/` (shipped PDFs + `gold.jsonl`).

Deps: `pymupdf`, `pandas`, `matplotlib`, `pillow`, `pytesseract` (+ tesseract binary), `imagehash`.

In [2]:
%pip install -q pymupdf pandas matplotlib pillow pytesseract imagehash

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, re, json, hashlib, random, io, collections, shutil, tempfile
import fitz, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import pytesseract, imagehash
%matplotlib inline

ROOT = os.getcwd()                                   # run from the "Milestone 2" folder
DATA = os.path.join(ROOT, "extracted", "Resumes PDF")
FIGS = os.path.join(ROOT, "figs"); os.makedirs(FIGS, exist_ok=True)
FD   = os.path.join(ROOT, "final_dataset")
RENDER_DIR = os.environ.get("RENDER_DIR", os.path.join(tempfile.gettempdir(), "gold_render"))
OCR_SAMPLE, OCR_DPI = 200, 200
PHASH_THRESH, GOLD_PER_CAT, DEV_FRAC = 6, 2, 0.4
random.seed(42)

## Helpers
`canon()` maps the 97 raw category folders to 43 canonical categories; `slug()`
makes filesystem-safe ids.

In [ ]:
def canon(folder: str) -> str:
    s = re.sub(r"[^a-z]", "", folder.lower().replace("resumes", ""))
    aliases = {
        "datascience":"Data Science","pythondeveloper":"Python Developer","javadeveloper":"Java Developer",
        "dotnetdeveloper":"DotNet Developer","dot":"DotNet Developer","dotdeveloper":"DotNet Developer",
        "reactdeveloper":"React Developer","react":"React Developer","sqldeveloper":"SQL Developer","sql":"SQL Developer",
        "sapdeveloper":"SAP Developer","etldeveloper":"ETL Developer","etl":"ETL Developer","devopsengineer":"DevOps Engineer",
        "mechanicalengineer":"Mechanical Engineer","electricalengineer":"Electrical Engineer","electricalengineering":"Electrical Engineer",
        "civilengineer":"Civil Engineer","businessanalyst":"Business Analyst","networksecurityengineer":"Network Security Engineer",
        "healthfitness":"Health & Fitness","foodbeverages":"Food & Beverages","food":"Food & Beverages",
        "buildingconstruction":"Building & Construction","webdesigning":"Web Designing","designing":"Web Designing",
        "design":"Designer","designer":"Designer","digitalmedia":"Digital Media","digital":"Digital Media",
        "humanresources":"Human Resources","hr":"Human Resources","informationtechnology":"Information Technology","it":"Information Technology",
        "publicrelations":"Public Relations","public":"Public Relations","operationmanager":"Operations Manager","operationsmanager":"Operations Manager",
        "managment":"Management","management":"Management","consult":"Consultant","consultant":"Consultant",
        "architects":"Architect","architect":"Architect","agriculture":"Agricultural","agricultural":"Agricultural",
        "sales":"Sales","finance":"Finance","banking":"Banking","accountant":"Accountant","advocate":"Advocate","arts":"Arts",
        "aviation":"Aviation","automobile":"Automobile","apparel":"Apparel","testing":"Testing","database":"Database",
        "education":"Education","blockchain":"Blockchain","bpo":"BPO","pmo":"PMO","pbo":"PMO","nse":"Network Security Engineer",
    }
    return aliases.get(s, folder.strip())

def slug(s): return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

## Part 1 — Exploratory Data Analysis

### 1.1 Per-file structural scan
One row per resume: category, size, page count, PDF text-layer chars, embedded
image count/resolution, filename family, and an MD5 (exact-duplicate key).
The resulting `df` is reused directly in Part 2 (no intermediate CSV).

In [ ]:
rows = []
for cf in sorted(os.listdir(DATA)):
    cdir = os.path.join(DATA, cf)
    if not os.path.isdir(cdir): continue
    for fn in os.listdir(cdir):
        if not fn.lower().endswith(".pdf"): continue
        fp = os.path.join(cdir, fn)
        rec = {"raw_folder": cf, "category": canon(cf), "file": fn,
               "size_bytes": os.path.getsize(fp),
               "name_family": "Image_N" if fn.lower().startswith("image_")
                              else ("numeric" if re.fullmatch(r"\d+\.pdf", fn) else "other")}
        doc = fitz.open(fp)
        rec["pages"] = doc.page_count
        txt = "".join(p.get_text() for p in doc)
        imgs, w, h = 0, 0, 0
        for p in doc:
            for im in p.get_images(full=True):
                imgs += 1; info = doc.extract_image(im[0]); w = max(w, info["width"]); h = max(h, info["height"])
        doc.close()
        rec.update(chars=len(txt.strip()), n_images=imgs, img_w=w, img_h=h,
                   megapixels=round(w*h/1e6, 2),
                   file_hash=hashlib.md5(open(fp, "rb").read()).hexdigest())
        rows.append(rec)
df = pd.DataFrame(rows)
print(len(df), "resumes scanned;", df.category.nunique(), "canonical categories")
df.head()

### 1.2 OCR text-length sample
No text layer exists, so text length is measured by OCR (Tesseract) on a
reproducible random sample of 200 resumes at 200 DPI.

In [ ]:
ocr = []
for fp in random.sample([os.path.join(DATA, r.raw_folder, r.file) for r in df.itertuples()], OCR_SAMPLE):
    try:
        d = fitz.open(fp); pix = d[0].get_pixmap(dpi=OCR_DPI); d.close()
        t = pytesseract.image_to_string(Image.open(io.BytesIO(pix.tobytes("png"))))
        ocr.append({"chars": len(t.strip()), "words": len(t.split())})
    except Exception:
        ocr.append({"chars": 0, "words": 0})
ocr = pd.DataFrame(ocr)
ocr.describe()

### 1.3 Summary numbers
The headline statistics cited in the report.

In [ ]:
cat = df["category"].value_counts()
summary = {
    "total_pdfs": int(len(df)), "distinct_raw_folders": int(df.raw_folder.nunique()),
    "distinct_canonical_categories": int(df.category.nunique()),
    "all_single_page": bool((df.pages == 1).all()), "all_image_only": bool((df.chars == 0).all()),
    "one_image_per_page": bool((df.n_images == df.pages).all()),
    "exact_duplicate_files": int(df.file_hash.duplicated().sum()),
    "name_family_counts": df.name_family.value_counts().to_dict(),
    "img_resolution_median": [int(df.img_w.median()), int(df.img_h.median())],
    "megapixels_median": float(df.megapixels.median()),
    "size_kb_median": round(df.size_bytes.median()/1024, 1),
    "category_max": {"name": cat.idxmax(), "n": int(cat.max())},
    "category_min": {"name": cat.idxmin(), "n": int(cat.min())},
    "imbalance_ratio": round(cat.max()/cat.min(), 1),
    "ocr_sample": {"n": OCR_SAMPLE, "dpi": OCR_DPI,
                   "words_median": int(ocr.words.median()), "words_p95": int(ocr.words.quantile(.95)),
                   "words_max": int(ocr.words.max()),
                   "approx_tokens_median": round(ocr.words.median()/0.75),
                   "approx_tokens_p95": round(ocr.words.quantile(.95)/0.75)},
}
print(json.dumps(summary, indent=2))

### 1.4 Figures
Saved to `figs/` (embedded in the report) and shown inline.

In [ ]:
plt.rcParams.update({"figure.dpi": 130, "font.size": 9})

top = cat.head(30)[::-1]
plt.figure(figsize=(8, 9)); plt.barh(top.index, top.values, color="#7a1f2b")
plt.title("Resumes by canonical job category (top 30)"); plt.xlabel("count")
plt.tight_layout(); plt.savefig(f"{FIGS}/01_category_distribution.png"); plt.show()

plt.figure(figsize=(5, 5))
plt.pie([0, len(df)], labels=["Text-extractable (0)", f"Image-only ({len(df)})"],
        autopct="%1.0f%%", colors=["#2a7a3b", "#c8862b"])
plt.title("Text-extractable vs image-only PDFs")
plt.tight_layout(); plt.savefig(f"{FIGS}/02_extractable_vs_ocr.png"); plt.show()

plt.figure(figsize=(6, 5)); plt.scatter(df.img_w, df.img_h, s=4, alpha=.25, color="#33556e")
plt.title(f"Image resolution (median {int(df.img_w.median())}x{int(df.img_h.median())})")
plt.xlabel("width px"); plt.ylabel("height px"); plt.xlim(0, 3000); plt.ylim(0, 5000)
plt.tight_layout(); plt.savefig(f"{FIGS}/03_image_resolution.png"); plt.show()

plt.figure(figsize=(6, 4)); plt.hist(df.size_bytes/1024, bins=60, color="#33556e")
plt.title("PDF file-size distribution"); plt.xlabel("KB"); plt.ylabel("count"); plt.xlim(0, 1200)
plt.tight_layout(); plt.savefig(f"{FIGS}/04_file_size.png"); plt.show()

## Part 2 — Dataset Preparation

### 2.1 Build ids
Add `path`, `stem`, and a stable `id` (`<category_slug>__<stem>`) to each row.

In [ ]:
df["path"] = [os.path.join(DATA, r.raw_folder, r.file) for r in df.itertuples()]
df["stem"] = df["file"].str.replace(r"\.pdf$", "", regex=True)
df["id"]   = [f"{slug(c)}__{s}" for c, s in zip(df.category, df.stem)]
df[["id", "category", "size_bytes"]].head()

### 2.2 Near-duplicate detection (perceptual hash)
Byte-level MD5 misses re-encoded near-duplicates. We pHash every resume (60 DPI),
bucket by hash prefix, and union-find group any pair within Hamming distance
`PHASH_THRESH`. This runs over all ~8,900 files (a few minutes).

In [ ]:
print(f"pHashing {len(df)} resumes ...")
ph = {}
for i, r in enumerate(df.itertuples()):
    try:
        d = fitz.open(r.path); pix = d[0].get_pixmap(dpi=60); d.close()
        ph[r.Index] = imagehash.phash(Image.open(io.BytesIO(pix.tobytes("png"))).convert("L"))
    except Exception:
        ph[r.Index] = None
    if (i + 1) % 1000 == 0: print(f"  {i+1}/{len(df)}", flush=True)

idx = [i for i in df.index if ph[i] is not None]
parent = {i: i for i in idx}
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
buckets = collections.defaultdict(list)
for i in idx: buckets[str(ph[i])[:4]].append(i)
for members in buckets.values():
    for a in range(len(members)):
        for b in range(a + 1, len(members)):
            if ph[members[a]] - ph[members[b]] <= PHASH_THRESH:
                parent[find(members[a])] = find(members[b])
groups = collections.defaultdict(list)
for i in idx: groups[find(i)].append(i)
gid = {m: k for k, ms in enumerate(groups.values()) for m in ms}
df["dup_group"] = df.index.map(lambda i: gid.get(i, -1))
n_groups = len(groups)
near_dup = sum(len(m) for m in groups.values() if len(m) > 1)
print(f"{n_groups} groups; {near_dup} files in near-duplicate groups")

### 2.3 Stratified gold sampling + dev/test split
Take one representative per duplicate group, sample `GOLD_PER_CAT` per category
(always keeping any already-annotated resumes), then split into dev/test.
Existing annotations in `final_dataset/gold.jsonl` are preserved.

In [ ]:
prev = {}
if os.path.exists(os.path.join(FD, "gold.jsonl")):
    for l in open(os.path.join(FD, "gold.jsonl")):
        r = json.loads(l)
        if r.get("_annotated"): prev[r["id"]] = r

reps = df.sort_values("size_bytes").groupby("dup_group").head(1)
gold_ids = []
for cat_name, grp in reps.groupby("category"):
    keep = [i for i in grp.index if df.at[i, "id"] in prev]          # always keep annotated
    others = [i for i in grp.sample(frac=1, random_state=42).index if i not in keep]
    gold_ids += (keep + others)[:max(GOLD_PER_CAT, len(keep))]
gold = df.loc[gold_ids].sample(frac=1, random_state=42).reset_index(drop=True)
n_dev = int(len(gold) * DEV_FRAC)
dev = set(gold["id"][:n_dev])
print(f"gold_total={len(gold)}  dev={len(dev)}  test={len(gold)-len(dev)}  categories={gold.category.nunique()}")

### 2.4 Build `final_dataset/`
Copy the gold PDFs (foldered by category), render a PNG per resume for
annotation (to `RENDER_DIR`, not shipped), and write `gold.jsonl` — merging any
existing annotations so re-running never wipes labelled work.

In [ ]:
shutil.rmtree(os.path.join(FD, "gold"), ignore_errors=True); os.makedirs(os.path.join(FD, "gold"))
shutil.rmtree(RENDER_DIR, ignore_errors=True); os.makedirs(RENDER_DIR)
SKELETON = {"contact": {"name": None, "email": None, "phone": None, "location": None, "links": []},
            "skills": [], "education": [], "experience": [], "projects": [], "certifications": [], "job_titles": []}
records = []
for r in gold.itertuples():
    cat_dir = os.path.join(FD, "gold", slug(r.category)); os.makedirs(cat_dir, exist_ok=True)
    shutil.copy2(r.path, os.path.join(cat_dir, f"{r.id}.pdf"))
    d = fitz.open(r.path); d[0].get_pixmap(dpi=150).save(os.path.join(RENDER_DIR, f"{r.id}.png")); d.close()
    rec = {"id": r.id, "category": r.category, "eval_split": "dev" if r.id in dev else "test",
           "pdf": f"gold/{slug(r.category)}/{r.id}.pdf",
           **json.loads(json.dumps(SKELETON))}
    if r.id in prev:
        for k in list(SKELETON) + ["contact"]: rec[k] = prev[r.id].get(k, rec[k])
        rec["_annotated"] = True
    records.append(rec)
with open(os.path.join(FD, "gold.jsonl"), "w") as f:
    for rec in records: f.write(json.dumps(rec, ensure_ascii=False) + "\n")
if os.path.exists(os.path.join(ROOT, "parsed_resume_schema.json")):
    shutil.copy2(os.path.join(ROOT, "parsed_resume_schema.json"), os.path.join(FD, "parsed_resume_schema.json"))

prep = {"corpus_total": int(len(df)), "near_dup_groups": int(n_groups),
        "near_duplicate_files": int(near_dup), "gold_per_category": GOLD_PER_CAT,
        "gold_total": int(len(gold)), "gold_dev": int(len(dev)), "gold_test": int(len(gold) - len(dev)),
        "gold_categories": int(gold.category.nunique()),
        "already_annotated": int(sum(1 for r in records if r.get("_annotated")))}
print(json.dumps(prep, indent=2)); print("render dir:", RENDER_DIR)